In [1]:
# Cell 1 — Config
import json, random, re
from pathlib import Path
import vertexai
from vertexai.batch_prediction import BatchPredictionJob
from google.cloud import storage

PROJECT_ID   = 'gen-lang-client-0944146289'
LOCATION     = 'us-central1'
BUCKET_NAME  = 'clones-batch-alvaro'
MODEL        = 'gemini-2.5-flash'
SAMPLE_RATIO = 0.10
RANDOM_SEED  = 42
MAX_LINES    = 500

DATASETS = {
    'dataset general':   {'clones': './dataset general/clones',   'js_files': './dataset general/js_files'},
    'dataset solutions': {'clones': './dataset solutions/clones', 'js_files': './dataset solutions/js_files'},
}

OUTPUT_FOLDER = './validation'

vertexai.init(project=PROJECT_ID, location=LOCATION)
storage_client = storage.Client(project=PROJECT_ID)
print('Config OK')


Config OK


In [2]:
# Cell 2 — Prompt de validacion
TYPE_DESCRIPTIONS = {
    1: 'Type I: identical except for whitespace and comments. All identifiers, logic and structure unchanged.',
    2: 'Type II: same structure and logic, but all identifier names renamed.',
    3: 'Type III: like Type II but with fragments added, removed or modified. Similar functionality.',
    4: 'Type IV: completely different syntax and implementation but equivalent functionality.',
}

VALIDATION_PROMPT = '''You are a code clone expert validator.
You receive a pair of JavaScript code snippets: an ORIGINAL and a CLONE with a declared clone type.
Decide if the clone is a valid and coherent example of its declared type.
Be lenient: minor imperfections are ok as long as the overall intent matches.

Type definitions:
- Type I: Identical except whitespace/comments. All identifiers unchanged.
- Type II: Same structure, all identifiers renamed.
- Type III: Like Type II but with fragments added/removed/modified. Similar functionality.
- Type IV: Completely different implementation but equivalent functionality.

Respond ONLY with a JSON object, no markdown:
{"id": "<id>", "valid": true or false, "reason": "one short sentence"}'''

print('Prompt OK')


Prompt OK


In [3]:
# Cell 3 — Preparar JSONL (10% aleatorio, un request por par)
random.seed(RANDOM_SEED)
requests = []
meta = []

for dataset_name, paths in DATASETS.items():
    clones_dir  = Path(paths['clones'])
    jsfiles_dir = Path(paths['js_files'])

    if not clones_dir.exists():
        print(f'[WARN] No existe: {clones_dir}')
        continue

    clone_files = list(clones_dir.glob('*_clon_*.js'))
    sample_size = max(1, int(len(clone_files) * SAMPLE_RATIO))
    sampled     = random.sample(clone_files, sample_size)
    print(f'{dataset_name}: {len(clone_files)} clones -> muestra {len(sampled)}')

    for clone_path in sampled:
        m = re.match(r'^(.+)_clon_(\d+)\.js$', clone_path.name)
        if not m:
            continue
        file_id    = m.group(1)
        clone_type = int(m.group(2))
        orig_path  = jsfiles_dir / f'{file_id}.js'

        if not orig_path.exists():
            print(f'  [WARN] original no encontrado: {orig_path}')
            continue

        try:
            original_code = orig_path.read_text(encoding='utf-8', errors='ignore')
            clone_code    = clone_path.read_text(encoding='utf-8', errors='ignore')
        except Exception as e:
            print(f'  [WARN] error leyendo {clone_path.name}: {e}')
            continue

        orig_lines  = original_code.splitlines()
        clone_lines = clone_code.splitlines()
        if len(orig_lines)  > MAX_LINES: original_code = '\n'.join(orig_lines[:MAX_LINES])
        if len(clone_lines) > MAX_LINES: clone_code    = '\n'.join(clone_lines[:MAX_LINES])

        custom_id   = f"{dataset_name.replace(' ', '_')}|{clone_path.name}"
        type_desc   = TYPE_DESCRIPTIONS[clone_type]
        prompt_text = (
            VALIDATION_PROMPT
            + f'\n\nPair ID: {custom_id}'
            + f'\nDeclared clone type: {clone_type} - {type_desc}'
            + f'\n\n[ORIGINAL]\n{original_code}'
            + f'\n\n[CLONE]\n{clone_code}'
        )

        requests.append({
            'custom_id': custom_id,
            'request': {
                'model': f'models/{MODEL}',
                'contents': [{'role': 'user', 'parts': [{'text': prompt_text}]}],
                'generation_config': {'temperature': 0.1}
            }
        })
        meta.append({'id': custom_id, 'dataset': dataset_name, 'file_id': file_id, 'clone_type': clone_type})

jsonl_path = Path('validation_requests.jsonl')
with open(jsonl_path, 'w', encoding='utf-8') as f:
    for req in requests:
        f.write(json.dumps(req) + '\n')

Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
with open(f'{OUTPUT_FOLDER}/meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f'\nTotal requests : {len(requests)}')
print(f'JSONL guardado : {jsonl_path}')
print(f'Meta guardado  : {OUTPUT_FOLDER}/meta.json')


dataset general: 9995 clones -> muestra 999
dataset solutions: 20000 clones -> muestra 2000

Total requests : 2999
JSONL guardado : validation_requests.jsonl
Meta guardado  : ./validation/meta.json


In [4]:
# Cell 4 — Subir a GCS y lanzar BatchPredictionJob
bucket    = storage_client.bucket(BUCKET_NAME)
gcs_input = 'validation/input/validation_requests.jsonl'
blob      = bucket.blob(gcs_input)
blob.upload_from_filename(str(jsonl_path))
print(f'Uploaded to: gs://{BUCKET_NAME}/{gcs_input}')

print('Submitting batch job...')
job = BatchPredictionJob.submit(
    source_model      = f'publishers/google/models/{MODEL}',
    input_dataset     = f'gs://{BUCKET_NAME}/{gcs_input}',
    output_uri_prefix = f'gs://{BUCKET_NAME}/validation/output/',
)
print(f'Job name  : {job.name}')
print(f'Job state : {job.state}')
print(f'\nGuarda este nombre para la Cell 5:')
print(f'  job_name = "{job.name}"')


Uploaded to: gs://clones-batch-alvaro/validation/input/validation_requests.jsonl
Submitting batch job...
BatchPredictionJob created. Resource name: projects/648630313845/locations/us-central1/batchPredictionJobs/4052122818517663744
To use this BatchPredictionJob in another session:
job = batch_prediction.BatchPredictionJob('projects/648630313845/locations/us-central1/batchPredictionJobs/4052122818517663744')
View Batch Prediction Job:
https://console.cloud.google.com/ai/platform/locations/us-central1/batch-predictions/4052122818517663744?project=648630313845
Job name  : 4052122818517663744
Job state : 2

Guarda este nombre para la Cell 5:
  job_name = "4052122818517663744"


In [10]:
# Cell 5 — Revisar estado (corre varias veces hasta SUCCEEDED)
job_name = '4052122818517663744'  # <-- pega aqui el job.name de la Cell 4

job = BatchPredictionJob(job_name)
job.refresh()
print(f'Estado: {job.state}')
print(f'Inicio: {job.create_time}')
if hasattr(job, 'end_time') and job.end_time:
    print(f'Fin      : {job.end_time}')
    print(f'Duracion : {job.end_time - job.create_time}')


Estado: 3
Inicio: 2026-04-29 01:22:53.823124+00:00


In [13]:
# Cell 6 — Descargar resultados y generar validation/results.json
job = BatchPredictionJob(job_name)
job.refresh()

state = str(job.state)
print(f'Estado: {state}')
if 'SUCCEEDED' not in state and job.state != 4:
    print('El job aun no termino.')
else:
    with open(f'{OUTPUT_FOLDER}/meta.json', 'r', encoding='utf-8') as f:
        meta_map = {m['id']: m for m in json.load(f)}

    bucket = storage_client.bucket(BUCKET_NAME)
    blobs  = list(bucket.list_blobs(prefix='validation/output/'))

    results, parse_errors = [], 0

    for blob in blobs:
        if not blob.name.endswith('.jsonl'):
            continue
        for line in blob.download_as_text().strip().split('\n'):
            if not line:
                continue
            try:
                data      = json.loads(line)
                custom_id = data.get('custom_id', 'unknown')
                raw_text  = data['response']['candidates'][0]['content']['parts'][0]['text']
                clean     = re.sub(r'^```[a-zA-Z]*\n?', '', raw_text.strip())
                clean     = re.sub(r'\n?```$', '', clean).strip()
                item      = json.loads(clean)
                if isinstance(item, list):
                    item = item[0]
                m = meta_map.get(custom_id, {})
                results.append({
                    'id': custom_id, 'dataset': m.get('dataset', ''),
                    'file_id': m.get('file_id', ''), 'clone_type': m.get('clone_type'),
                    'valid': item.get('valid'), 'reason': item.get('reason', ''),
                })
            except Exception as e:
                parse_errors += 1
                results.append({'id': 'unknown', 'dataset': '', 'file_id': '', 'clone_type': None, 'valid': None, 'reason': f'parse_error: {e}'})

    results_path = Path(OUTPUT_FOLDER) / 'results.json'
    with open(results_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f'Resultados guardados : {results_path}')
    print(f'Total procesados     : {len(results)}')
    print(f'Errores de parseo    : {parse_errors}')


Estado: 4
Resultados guardados : validation/results.json
Total procesados     : 2999
Errores de parseo    : 1


In [14]:
# Cell 7 — Resumen final
with open(f'{OUTPUT_FOLDER}/results.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

print('=' * 55)
print('RESUMEN DE VALIDACION')
print('=' * 55)

for ds in ['dataset general', 'dataset solutions']:
    ds_res = [r for r in results if r['dataset'] == ds]
    if not ds_res:
        continue
    print(f'\n{ds}:')
    for t in [1, 2, 3, 4]:
        t_res = [r for r in ds_res if r['clone_type'] == t]
        if not t_res:
            continue
        valid = sum(1 for r in t_res if r.get('valid') is True)
        total = len(t_res)
        print(f'  Type {t}: {valid}/{total} validos ({valid/total*100:.1f}%)')

total_valid = sum(1 for r in results if r.get('valid') is True)
total_all   = len(results)
print(f'\nTOTAL: {total_valid}/{total_all} validos ({total_valid/total_all*100:.1f}%)')


RESUMEN DE VALIDACION

dataset general:
  Type 1: 207/271 validos (76.4%)
  Type 2: 201/257 validos (78.2%)
  Type 3: 206/208 validos (99.0%)
  Type 4: 256/263 validos (97.3%)

dataset solutions:
  Type 1: 438/477 validos (91.8%)
  Type 2: 439/512 validos (85.7%)
  Type 3: 495/502 validos (98.6%)
  Type 4: 483/508 validos (95.1%)

TOTAL: 2725/2999 validos (90.9%)
